In [ ]:
# Body measurement validation notebook

This notebook validates the real image-based measurement pipeline requested for StyleDNA: a MediaPipe segmentation-based approach, explicit image-quality rejection boundaries, and width calculations from the body mask.
</VSCode.Cell>
<VSCode.Cell language="python">
import io
from types import SimpleNamespace

import numpy as np
from PIL import Image, ImageDraw

from app.services.body.measure import VisualMeasurementError, _count_connected_components, _measure_band_width, _validate_pose_and_mask, estimate_proportions_from_image
from app.services.body.shape import classify_body_shape

print('imports ok')
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 1. Inspect existing measurement pipeline and target function

We review the imported service and confirm the target contract: the visual measurement function should return shoulder, waist, and hip widths and raise `VisualMeasurementError` for invalid input rather than falling back to a fake value.
</VSCode.Cell>
<VSCode.Cell language="python">
# Smoke-check the public contract
print(estimate_proportions_from_image.__name__)
print(VisualMeasurementError.__name__)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 2. Review body rules and shape metadata inputs

The body rules YAML defines the ratios that map shoulder/waist/hip widths into body-shape categories. We validate that the estimator produces width values that remain compatible with the shape classifier.
</VSCode.Cell>
<VSCode.Cell language="python">
# Example canonical shape profiles
for shape_name, params in {
    'Hourglass': {'shoulder': 95, 'waist': 66, 'hip': 96},
    'Pear': {'shoulder': 84, 'waist': 68, 'hip': 102},
    'Rectangle': {'shoulder': 92, 'waist': 80, 'hip': 94},
    'Apple': {'shoulder': 94, 'waist': 92, 'hip': 95},
}.items():
    outcome = classify_body_shape(**params)
    print(shape_name, outcome['shape'], outcome['confidence'])
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 3. Analyze route contract and expected error behavior

The API should surface `VisualMeasurementError` as a validation error and must reject malformed, cropped, side-view, or low-quality body photos before storing or returning a body profile.
</VSCode.Cell>
<VSCode.Cell language="python">
class FakeLandmark(SimpleNamespace):
    pass

# side-view rejection scenario
mask = np.ones((460, 220), dtype=bool)
landmarks = {
    'left_shoulder': FakeLandmark(x=0.12, y=0.22, visibility=1.0),
    'right_shoulder': FakeLandmark(x=0.42, y=0.22, visibility=1.0),
    'left_hip': FakeLandmark(x=0.18, y=0.56, visibility=1.0),
    'right_hip': FakeLandmark(x=0.50, y=0.56, visibility=1.0),
}
try:
    _validate_pose_and_mask(mask, landmarks, 220, 460)
    print('unexpected pass')
except VisualMeasurementError as exc:
    print(type(exc).__name__, str(exc))
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 4. Implement MediaPipe PoseLandmarker segmentation workflow

The modern API uses ` mediapipe.tasks.python.vision.PoseLandmarker ` and `ImageSegmenter` to create a binary mask from the subject silhouette while ensuring dependency failures surface as clear errors.
</VSCode.Cell>
<VSCode.Cell language="python">
# This cell documents the expected integration point rather than invoking heavy model downloads.
# The actual implementation is in app/services/body/measure.py.
print('PoseLandmarker + segmentation workflow is defined in the body measurement service.')
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 5. Validate image framing and pose quality before measuring

We enforce: no cropped body; no side-view pose; no multiple figures; no poor visibility; and enough image dimensions for a valid front-facing portrait shot.
</VSCode.Cell>
<VSCode.Cell language="python">
mask = np.zeros((460, 220), dtype=bool)
mask[50:410, 20:200] = True
landmarks = {
    'left_shoulder': FakeLandmark(x=0.2, y=0.2, visibility=1.0),
    'right_shoulder': FakeLandmark(x=0.3, y=0.2, visibility=1.0),
    'left_hip': FakeLandmark(x=0.2, y=0.4, visibility=1.0),
    'right_hip': FakeLandmark(x=0.3, y=0.4, visibility=1.0),
}
try:
    _validate_pose_and_mask(mask, landmarks, 220, 460)
    print('expected validation pass with a valid mask')
except VisualMeasurementError as exc:
    print('validation failed unexpectedly:', exc)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 6. Measure shoulder, waist, and hip widths from the segmentation mask

The mask-based measurement algorithm scans each body band and uses the maximum or minimum width of the subject silhouette to estimate the shoulder row, hip band, and waist band values.
</VSCode.Cell>
<VSCode.Cell language="python">
width, height = 220, 460
mask = np.zeros((height, width), dtype=bool)
center_x = width // 2
for y in range(height):
    if 100 <= y <= 120:
        mask[y, center_x - 50:center_x + 50] = True
    elif 200 <= y <= 240:
        mask[y, center_x - 35:center_x + 35] = True
    elif 280 <= y <= 320:
        mask[y, center_x - 60:center_x + 60] = True
print('shoulder band width', _measure_band_width(mask, 110, 0.08, False))
print('hip band width', _measure_band_width(mask, 300, 0.10, False))
print('waist band width', _measure_band_width(mask, 220, 0.05, True))
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 7. Remove constant fallback and surface missing-model errors clearly

If MediaPipe is unavailable or the model assets are missing, the code raises `VisualMeasurementError` with a clear remediation message instead of silently substituting a fake body measurement.
</VSCode.Cell>
<VSCode.Cell language="python">
print('Missing model behavior is explicit: raise VisualMeasurementError with installation guidance.')
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 8. Add Pillow silhouette tests for valid shape profiles

The unit tests now generate body silhouettes in pillow and validate that width-based classifications return the expected shape names for each canonical profile.
</VSCode.Cell>
<VSCode.Cell language="python">
# Build a valid silhouette image as a test fixture.
img = Image.new('RGB', (220, 460), color=(255, 255, 255))
draw = ImageDraw.Draw(img)
draw.ellipse((100, 20, 120, 40), fill=(0, 0, 0))
draw.rectangle((70, 100, 150, 260), fill=(0, 0, 0))
for left, right in [(60, 160), (80, 140), (90, 130)]:
    draw.rectangle((left, 100, right, 180), fill=(0, 0, 0))

buf = io.BytesIO()
img.save(buf, format='PNG')
print('silhouette bytes:', len(buf.getvalue()))
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 9. Add reject-case tests for invalid visual conditions

The reject-case tests cover low-visibility masks, side-view poses, and cropped or multi-person scenes, each of which must raise `VisualMeasurementError`.
</VSCode.Cell>
<VSCode.Cell language="python">
# Negative scenarios are validated in the shape measurement unit tests.
print('Reject-case checks are covered in tests/unit/test_body.py')
